# Action Recognition in Videos — UCF101 (Kaggle, v3)

Full 101-class pipeline: class-weighted loss, early stopping, bounded
hyperparameter search, and a step-by-step Streamlit frontend (run
locally after downloading results).

Repo: `Action-Recognition-in-Videos-ft-SRINATH-clzv3`

**Before running:** in the right-hand sidebar —
1. Settings → Accelerator → **GPU T4 x2** (or P100)
2. Settings → Internet → **On**

In [ ]:
!nvidia-smi

In [ ]:
# Clone and auto-detect the correct project folder, regardless of nesting.
import os

os.makedirs("/kaggle/working", exist_ok=True)
os.chdir("/kaggle/working")
if not os.path.isdir("Action-Recognition-in-Videos-ft-SRINATH-clzv3"):
    !git clone https://github.com/SrinathRavi10/Action-Recognition-in-Videos-ft-SRINATH-clzv3.git
os.chdir("/kaggle/working/Action-Recognition-in-Videos-ft-SRINATH-clzv3")

if not os.path.isdir("src"):
    found = False
    for root, dirs, files in os.walk("."):
        if "src" in dirs and os.path.isfile(os.path.join(root, "src", "config.py")):
            os.chdir(root)
            found = True
            break
    if not found:
        raise RuntimeError("Could not find a folder containing src/config.py — check the repo structure manually.")

print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))
assert os.path.isdir("src") and os.path.isfile("requirements.txt"), "Still not in the right folder!"
print("\nConfirmed: src/ and requirements.txt found. Ready to proceed.")

In [ ]:
# Kaggle pre-installs torch/torchvision/opencv/sklearn/matplotlib — this mostly
# just adds mediapipe, huggingface_hub, streamlit, plotly.
!pip install -q -r requirements.txt

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Download the full 101-class dataset

Expect this to take a while — it's ~10x the data of a small subset.

In [ ]:
!python data/download_ucf101.py

## 2. (Optional) Bounded hyperparameter probe

Not exhaustive — see the README's Precautions table for why.

In [ ]:
!python src/hyperparam_search.py

In [ ]:
import json
with open("outputs/best_hparams.json") as f:
    hp = json.load(f)
print("Best probe config:", hp["best"])

## 3. Train

In [ ]:
!python src/train.py

## 4. Evaluate

In [ ]:
!python src/evaluate.py --checkpoint checkpoints/best_model.pt

In [ ]:
from IPython.display import Image, display
display(Image("outputs/confusion_matrix.png"))

## 5. Visualize predictions

In [ ]:
!python src/visualize_predictions.py --checkpoint checkpoints/best_model.pt --num_samples 8

from IPython.display import Image, display
display(Image("outputs/sample_predictions.png"))

## 6. Pose-overlay video visualizations

In [ ]:
!python src/visualize_pose_predictions.py --checkpoint checkpoints/best_model.pt --num_clips 5

## 7. Package results for the local frontend

The Streamlit frontend runs on your own machine, not inline in Kaggle.
This zips everything it needs into one file you can download from the
file browser (folder icon, left sidebar) or via 'Save Version'.

In [ ]:
!zip -r /kaggle/working/frontend_artifacts.zip checkpoints/best_model.pt outputs data/UCF101_subset/classes.txt data/UCF101_subset/class_distribution.json
print("Saved to /kaggle/working/frontend_artifacts.zip — download it from the file browser.")